# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = json.loads(dataset.metadata.json())
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entity references use their `@id`.

Let's enumerate the available record sets and their fields/columns:

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print('No record sets found in the dataset.')
else:
    for i, rs in enumerate(record_sets):
        print(f"[{i}] RecordSet @id: {rs.id}, name: {rs.name}")
        print("  Fields/columns:")
        for field in rs.fields:
            col_ids = getattr(field, 'columns', [])
            print(f"     - Field @id: {field.id}, name: {field.name}, columns: {[c.id for c in col_ids] if col_ids else 'N/A'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note**: If there are multiple record sets, we'll extract them into pandas DataFrames dynamically and inspect their content by referencing their `@id`.

In [ ]:
# Prepare to extract records from each record set
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load records for each record set and store in a dataframe
for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id}. Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Failed to load records for RecordSet {record_set_id}: {e}")

# Choose a main record set for further analysis (as an example we take the first one, if any)
if all_record_set_ids:
    main_record_set_id = all_record_set_ids[0]
    print(f"Using main RecordSet @id: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print(main_df.info())
else:
    main_record_set_id = None
    main_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Tip:** Reference schema fields/columns by their `@id` as shown above.

Let's select a numeric field from the schema for filtered analysis. We'll automatically pick the first numeric column if any.

In [ ]:
import numpy as np

numeric_field_id = None
group_field_id = None

if main_record_set_id is not None and not main_df.empty:
    # Attempt to retrieve numeric columns from the schema field definitions
    record_set = next(rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id)
    for field in record_set.fields:
        # Try to determine if field is numeric by dataType
        dtype = getattr(field, 'data_type', None)
        if dtype is not None:
            # Accept 'Float', 'Integer', or 'Number', schema-style names
            if dtype.lower() in ['float', 'integer', 'number', 'double']:
                if field.id in main_df.columns:
                    numeric_field_id = field.id
                    break
    # Pick a group field (categorical)
    for field in record_set.fields:
        dtype = getattr(field, 'data_type', None)
        # Accept 'Text', 'String', or others
        if dtype is not None:
            if dtype.lower() in ['text', 'string', 'category', 'categorical'] and field.id in main_df.columns:
                group_field_id = field.id
                break
    # If not found, fallback to first numeric/text column
    if numeric_field_id is None:
        for col in main_df.columns:
            if np.issubdtype(main_df[col].dtype, np.number):
                numeric_field_id = col
                break
    if group_field_id is None:
        for col in main_df.columns:
            if main_df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break

    print(f"Selected numeric field for analysis: {numeric_field_id}")
    print(f"Selected group (categorical) field: {group_field_id}")

    if numeric_field_id is not None:
        # Filter to values above threshold
        threshold = main_df[numeric_field_id].mean() if np.issubdtype(main_df[numeric_field_id].dtype, np.number) else 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (z-score):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by a categorical field (if present)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id}:")
            print(grouped_df.head())
    else:
        print('No numeric field found for EDA in this record set.')
else:
    print('No main record set dataframe loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id in main_df.columns:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If a group field exists, show boxplot
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found to visualize.')

## 6. Conclusion
In this notebook, we explored the FAIR² dataset describing mass spectrometry results for extracellular vesicles from human mast cells. We loaded and browsed its schema and record set structure, referenced all relevant fields by their `@id`, loaded records, performed basic EDA (filtering, normalization, grouping), and visualized selected attributes. This approach can be adapted for other Croissant-compatible datasets for robust FAIR data analysis and sharing.

Feel free to customize this notebook for further downstream protein analysis or integration into biomedical pipelines.